# Retrieval-Augmented Generation Document QA System

## Objective

Build and measure an end-to-end document-QA RAG system on the public
**RAG Mini-Wikipedia** benchmark. The experiment compares a dense-retrieval
baseline with an advanced pipeline using HyDE, dense+sparse reciprocal-rank
fusion, cross-encoder reranking, evidence-first prompting, and citations.

Both systems use the same local Llama answer model, isolating the value of
the retrieval and prompting changes. The final cells compute real metrics,
save per-example evidence, and provide optional GPT-4o and DeepInfra adapters.


In [ ]:
# Select the isolated, hash-locked inference environment. GPU execution is deferred.
import os, sys
from pathlib import Path
from importlib.metadata import version
REPO_ROOT = Path(os.environ.get("PROJECT_REPO", Path.cwd())).resolve()
sys.path.insert(0, str(REPO_ROOT / "tools"))
from training_environment import validate
print(validate(REPO_ROOT / "tools/requirements-inference-v2.hashes.txt"))


In [ ]:
import gc
import json
import math
import os
import pathlib
import random
import re
import statistics
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch

# Standard HTTP downloads are more reliable through the VS Code Colab proxy.
os.environ.setdefault('HF_HUB_DISABLE_XET', '1')

PROJECT_DIR = pathlib.Path(os.environ.get('PORTFOLIO_PROJECT_DIR', '/content/rag_document_qa'))
CACHE_DIR = PROJECT_DIR / 'cache'
from datetime import datetime, timezone
RUN_ID = os.environ.get('PORTFOLIO_RUN_ID') or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT_DIR = PROJECT_DIR / 'outputs/runs/upgrade-v2' / RUN_ID
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Seed 42 was used for prompt development; 2026 is the untouched final holdout.
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)

DATASET_NAME = 'rag-datasets/rag-mini-wikipedia'
DATASET_REVISION = 'ec8e240df1b1eb7171cb3e5dcf9462f6569f4544'
EMBED_MODEL = 'intfloat/e5-small-v2'
RERANK_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
GENERATOR_MODEL = 'meta-llama/Meta-Llama-3-8B-Instruct'
EVAL_SIZE = 60
BASELINE_K = 5
ADVANCED_K = 5
CANDIDATE_K = 20

if not torch.cuda.is_available():
    raise RuntimeError('A CUDA runtime is required. Select the connected A100 Colab kernel.')
DEVICE = torch.device('cuda:0')
print('GPU:', torch.cuda.get_device_name(0))

# Release large Project 1 models if both notebooks share the same Colab runtime.
for variable_name in ['model', 'base_model', 'trainer']:
    if variable_name in globals():
        del globals()[variable_name]
gc.collect()
torch.cuda.empty_cache()


## 1. Load and cache a real benchmark

The dataset contains roughly 3,200 Wikipedia passages and 918 questions.
It is public (CC BY 3.0), small enough for a portfolio notebook, and large
enough to exercise real retrieval. A pinned dataset revision makes later
runs reproducible. `save_to_disk` means the network download happens once.


In [ ]:
from datasets import load_dataset, load_from_disk
from huggingface_hub import hf_hub_download, list_repo_files

CORPUS_CACHE = CACHE_DIR / f'mini_wikipedia_corpus_{DATASET_REVISION}'
QA_CACHE = CACHE_DIR / f'mini_wikipedia_qa_{DATASET_REVISION}'

if CORPUS_CACHE.exists() and QA_CACHE.exists():
    corpus_ds = load_from_disk(str(CORPUS_CACHE))
    qa_ds = load_from_disk(str(QA_CACHE))
    source = 'local cache'
else:
    repo_files = list_repo_files(
        DATASET_NAME, repo_type='dataset', revision=DATASET_REVISION
    )
    corpus_files = sorted(
        path for path in repo_files
        if path.startswith('text-corpus/passages/') and path.endswith('.parquet')
    )
    qa_files = sorted(
        path for path in repo_files
        if path.startswith('question-answer/test/') and path.endswith('.parquet')
    )
    if not corpus_files or not qa_files:
        raise RuntimeError('Could not locate the pinned corpus and QA parquet files.')

    def download_parquet_files(paths):
        return [
            hf_hub_download(
                repo_id=DATASET_NAME,
                filename=path,
                repo_type='dataset',
                revision=DATASET_REVISION,
                cache_dir=str(CACHE_DIR / 'huggingface'),
            )
            for path in paths
        ]

    corpus_ds = load_dataset(
        'parquet', data_files=download_parquet_files(corpus_files), split='train'
    )
    qa_ds = load_dataset(
        'parquet', data_files=download_parquet_files(qa_files), split='train'
    )
    corpus_ds.save_to_disk(str(CORPUS_CACHE))
    qa_ds.save_to_disk(str(QA_CACHE))
    source = 'Hugging Face download (now cached)'

print('Loaded from:', source)
print('Corpus rows:', len(corpus_ds), '| QA rows:', len(qa_ds))
print('Corpus columns:', corpus_ds.column_names)
print('QA columns:', qa_ds.column_names)


In [4]:
from langchain_core.documents import Document

documents = []
for row_index, row in enumerate(corpus_ds):
    passage = str(row.get('passage', '')).strip()
    if not passage:
        continue
    passage_id = str(row.get('id', row_index))
    documents.append(
        Document(
            page_content=passage,
            metadata={'source': 'mini_wikipedia', 'passage_id': passage_id},
        )
    )

assert documents, 'No corpus documents were created.'
print('LangChain documents:', len(documents))
print(documents[0].metadata, documents[0].page_content[:180])


LangChain documents: 3200
{'source': 'mini_wikipedia', 'passage_id': '0'} Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part of South America.  It is home to 3.3 million people, of which


## 2. Structure-aware chunking

Recursive splitting prefers paragraph and sentence boundaries before hard
character cuts. A 900-character window retains enough factual context while
a 120-character overlap protects facts near boundaries. Stable chunk IDs
support traceable citations and repeatable evaluation.


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=120,
    separators=['\n\n', '\n', '. ', ' ', ''],
)

chunks = []
for document in documents:
    for local_index, chunk in enumerate(splitter.split_documents([document])):
        chunk.metadata['chunk_id'] = (
            f"wiki::{document.metadata['passage_id']}::{local_index}"
        )
        chunks.append(chunk)

lengths = [len(chunk.page_content) for chunk in chunks]
print('Chunks:', len(chunks))
print('Characters per chunk: median', int(np.median(lengths)), 'p95', int(np.percentile(lengths, 95)))
assert len({chunk.metadata['chunk_id'] for chunk in chunks}) == len(chunks)


Chunks: 3492
Characters per chunk: median 305 p95 843


## 3. Build dense and sparse indexes

E5-small-v2 is a strong, efficient retrieval encoder. FAISS provides exact
inner-product search over normalized vectors, while BM25 captures names,
numbers, and rare terms that dense retrieval can miss. Embeddings are cached
so reruns skip the most expensive indexing step.


In [ ]:
import faiss
import torch.nn.functional as F
from rank_bm25 import BM25Okapi
from transformers import AutoModel, AutoTokenizer

def lexical_tokens(text):
    return re.findall(r"[a-z0-9]+", text.lower())

class E5Encoder:
    def __init__(self, model_name, device):
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, revision=EMBED_REVISION)
        self.model = AutoModel.from_pretrained(model_name, revision=EMBED_REVISION).to(device).eval()

    @torch.inference_mode()
    def encode(self, texts, prefix, batch_size=64):
        vectors = []
        for start in range(0, len(texts), batch_size):
            batch = [prefix + text for text in texts[start:start + batch_size]]
            encoded = self.tokenizer(
                batch, padding=True, truncation=True, max_length=512, return_tensors='pt'
            ).to(self.device)
            hidden = self.model(**encoded).last_hidden_state
            mask = encoded['attention_mask'].unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
            vectors.append(F.normalize(pooled, p=2, dim=1).float().cpu().numpy())
        return np.concatenate(vectors).astype('float32')

from huggingface_hub import HfApi
import hashlib
EMBED_REVISION = HfApi().model_info(EMBED_MODEL).sha
encoder = E5Encoder(EMBED_MODEL, DEVICE)
chunk_texts = [chunk.page_content for chunk in chunks]
embedding_spec = {"model": EMBED_MODEL, "revision": EMBED_REVISION,
                  "texts": chunk_texts, "prefix": "passage: ", "pooling": "mean",
                  "normalized": True, "max_length": 512,
                  "chunk_size": 900, "chunk_overlap": 120}
embedding_fingerprint = hashlib.sha256(json.dumps(embedding_spec, sort_keys=True).encode()).hexdigest()
embedding_cache = CACHE_DIR / f"embeddings_{embedding_fingerprint}.npy"

if embedding_cache.exists():
    chunk_embeddings = np.load(embedding_cache)
    if chunk_embeddings.shape[0] != len(chunks):
        raise RuntimeError('Cached embeddings do not match the current chunk count.')
    print('Loaded embeddings from cache:', embedding_cache)
else:
    chunk_embeddings = encoder.encode(chunk_texts, prefix='passage: ')
    np.save(embedding_cache, chunk_embeddings)
    print('Created and cached embeddings:', embedding_cache)

dense_index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
dense_index.add(chunk_embeddings)
tokenized_corpus = [lexical_tokens(text) for text in chunk_texts]
bm25 = BM25Okapi(tokenized_corpus)
print('FAISS vectors:', dense_index.ntotal, '| dimensions:', dense_index.d)


## 4. Shared local answer model

The connected runtime already contains the Meta Llama 3 8B weights used by
Project 1. Loading those cached weights in NF4 4-bit mode avoids another
multi-gigabyte transfer and provides a stronger local generator. Using one
generator for both systems controls a major confounder: score differences
can be attributed to retrieval and prompt design rather than model size.


In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

GENERATOR_REVISION = '8afb486c1db24fe5011ec46dfbe5b5dccdb575c2'
rag_generation_usage = []
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
generator_quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

try:
    generator_tokenizer = AutoTokenizer.from_pretrained(
        GENERATOR_MODEL, local_files_only=True, revision=GENERATOR_REVISION
    )
    generator_model = AutoModelForCausalLM.from_pretrained(
        GENERATOR_MODEL,
        local_files_only=True, revision=GENERATOR_REVISION,
        quantization_config=generator_quantization,
        device_map='auto',
        dtype=compute_dtype,
        attn_implementation='sdpa',
        trust_remote_code=False,
        use_safetensors=True,
    ).eval()
except OSError as exc:
    raise RuntimeError(
        'The cached Project 1 Llama weights were not found. Run Project 1 in this '
        'runtime or pre-download the model in the Colab UI, then rerun Cell 13.'
    ) from exc

if generator_tokenizer.pad_token_id is None:
    generator_tokenizer.pad_token = generator_tokenizer.eos_token
generator_tokenizer.padding_side = 'left'
generator_model.config.use_cache = True

@torch.inference_mode()
def generate_batch(prompts, max_new_tokens=96, batch_size=8):
    answers = []
    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start:start + batch_size]
        rendered = [
            generator_tokenizer.apply_chat_template(
                [
                    {'role': 'system', 'content': 'You are a precise document QA assistant.'},
                    {'role': 'user', 'content': prompt},
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
            for prompt in batch_prompts
        ]
        encoded = generator_tokenizer(
            rendered, padding=True, truncation=False, return_tensors='pt'
        ).to(DEVICE)
        if encoded['input_ids'].shape[1] > 3072:
            raise ValueError("RAG input exceeds the matched 3072-token budget; no silent truncation")
        generated = generator_model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=generator_tokenizer.pad_token_id,
        )
        new_tokens = generated[:, encoded['input_ids'].shape[1]:]
        for row_index in range(len(batch_prompts)):
            padding = (new_tokens[row_index] == generator_tokenizer.pad_token_id).nonzero()
            output_length = int(padding[0]) + 1 if len(padding) else new_tokens.shape[1]
            rag_generation_usage.append({
                'input_tokens': int(encoded['attention_mask'][row_index].sum()),
                'output_tokens': output_length})
        answers.extend(generator_tokenizer.batch_decode(new_tokens, skip_special_tokens=True))
    return [answer.strip() for answer in answers]

print('Loaded generator:', GENERATOR_MODEL)


## 5. Baseline and advanced retrieval

The baseline uses a raw-question E5 search. The advanced retriever creates a
hypothetical answer passage (HyDE), searches both dense and BM25 indexes,
combines rankings with reciprocal-rank fusion, and reranks the top candidates
with a query-document cross-encoder. This stack balances semantic recall,
exact-term recall, and final precision.


In [ ]:
from transformers import AutoModelForSequenceClassification

RERANK_REVISION = HfApi().model_info(RERANK_MODEL).sha
rerank_tokenizer = AutoTokenizer.from_pretrained(RERANK_MODEL, revision=RERANK_REVISION)
rerank_model = AutoModelForSequenceClassification.from_pretrained(RERANK_MODEL, revision=RERANK_REVISION).to(DEVICE).eval()

def dense_rank(query, k):
    query_vector = encoder.encode([query], prefix='query: ')
    scores, indices = dense_index.search(query_vector, k)
    return [(int(index), float(score)) for index, score in zip(indices[0], scores[0])]

def sparse_rank(query, k):
    scores = bm25.get_scores(lexical_tokens(query))
    indices = np.argsort(scores)[::-1][:k]
    return [(int(index), float(scores[index])) for index in indices]

def reciprocal_rank_fusion(rankings, limit, rrf_k=60):
    fused = Counter()
    for ranking in rankings:
        for rank, (index, _) in enumerate(ranking, start=1):
            fused[index] += 1.0 / (rrf_k + rank)
    return [index for index, _ in fused.most_common(limit)]

@torch.inference_mode()
def rerank(question, candidate_indices, k):
    pairs = [[question, chunk_texts[index]] for index in candidate_indices]
    encoded = rerank_tokenizer(
        pairs, padding=True, truncation=True, max_length=512, return_tensors='pt'
    ).to(DEVICE)
    scores = rerank_model(**encoded).logits.squeeze(-1).float().cpu().numpy()
    order = np.argsort(scores)[::-1][:k]
    return [candidate_indices[int(position)] for position in order]

def baseline_retrieve(question):
    return [index for index, _ in dense_rank(question, BASELINE_K)]

def advanced_retrieve(question, hypothetical_passage):
    dense_query = f"{question} {hypothetical_passage}"
    candidates = reciprocal_rank_fusion(
        [dense_rank(dense_query, CANDIDATE_K), sparse_rank(question, CANDIDATE_K)],
        limit=CANDIDATE_K,
    )
    return rerank(question, candidates, ADVANCED_K)

def format_context(indices):
    return '\n\n'.join(
        f"[{chunks[index].metadata['chunk_id']}] {chunks[index].page_content}"
        for index in indices
    )

def hyde_prompt(question):
    return (
        'Write a short Wikipedia-style passage that would answer the question below. '
        'Include likely entities, dates, and terminology. Return only the hypothetical passage.\n\n'
        f'Question: {question}'
    )

def baseline_answer_prompt(question, indices):
    return (
        'Answer the question using only the supplied context. If the answer is absent, say '
        '"Insufficient evidence." Give a concise answer and cite supporting chunk IDs in brackets.\n\n'
        f'Question: {question}\n\nContext:\n{format_context(indices)}'
    )

# Both arms use the exact same answer instruction.
advanced_answer_prompt = baseline_answer_prompt

print('Retriever components ready: dense, BM25, RRF, HyDE, cross-encoder reranker')


## 6. Reproducible evaluation

The official QA split does not provide gold passage IDs, so retrieval is
evaluated with **answer-hit@k**: whether a normalized reference answer occurs
in any retrieved chunk. We also report reciprocal rank, answer token F1,
citation validity, and latency. The deterministic sample only includes
questions whose answer text occurs somewhere in the supplied corpus.

The transparent composite score weights answer F1 (45%), answer-hit (25%),
reciprocal rank (15%), and citation validity (15%). The claimed improvement
is computed from saved predictions; no target percentage is hardcoded.

The v2 comparison adds conditional HyDE as a fifth arm. It skips hypothesis
generation when at least two of the top three dense and positive-scoring lexical
candidates agree. This preregistered rule is a heuristic, not calibrated relevance
or factual confidence. All arms retain the same answer prompt and final context
budget; the probe and optional hypothesis count toward measured latency and tokens.
Warm-up decisions are excluded. New model-backed results are pending.


In [ ]:
# Shared helpers are embedded so this notebook remains standalone in Colab.
"""CPU-testable measurement helpers, embedded into the standalone notebooks."""

import hashlib
import json
import math
import random
import re
import statistics
import time
from collections import Counter


def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=True).encode()).hexdigest()


def paired_mean_interval(before, after, seed=2026, resamples=2000):
    if not before or len(before) != len(after):
        raise ValueError("Paired evaluation requires nonempty, equally sized samples")
    if resamples < 40 or any(not math.isfinite(value) for value in [*before, *after]):
        raise ValueError("At least 40 resamples and finite observations are required")
    delta = [right-left for left, right in zip(before, after)]
    rng = random.Random(seed)
    means = sorted(statistics.fmean(rng.choices(delta, k=len(delta))) for _ in range(resamples))
    return {"mean_change": statistics.fmean(delta),
            "ci95": [means[int(.025*resamples)], means[min(resamples-1, int(.975*resamples))]],
            "pairs": len(delta), "resamples": resamples}


def normalized_words(text):
    return re.findall(r"[a-z0-9]+", re.sub(r"\[wiki::[^\]]+\]", " ", text.lower()))


def overlap_f1(answer, reference):
    predicted, gold = normalized_words(answer), normalized_words(reference)
    if not predicted or not gold:
        return float(predicted == gold)
    common = sum((Counter(predicted) & Counter(gold)).values())
    return 2*common/(len(predicted)+len(gold))


def rag_measurements(example, answer, retrieved_ids, corpus):
    reference = example["reference"]
    target = " ".join(normalized_words(reference))
    hits = [rank for rank, item in enumerate(retrieved_ids, 1)
            if target and f" {target} " in " " + " ".join(normalized_words(corpus[item])) + " "]
    retrieval_proxy_defined = target not in ("yes", "no") and example.get("answerable") is not False
    cited = set(re.findall(r"\[(wiki::[^\]]+)\]", answer))
    relevant = set(example.get("relevant_chunk_ids", []))
    binary = re.match(r"^\s*(yes|no)\b", answer.lower())
    return {
        "answer_hit_proxy": float(bool(hits)) if retrieval_proxy_defined else None,
        "reciprocal_rank_proxy": (1/hits[0] if hits else 0.0) if retrieval_proxy_defined else None,
        "token_f1": overlap_f1(answer, reference),
        "binary_answer_correct": float(bool(binary) and binary.group(1) == target) if target in ("yes", "no") else None,
        "citation_id_validity": len(cited & set(retrieved_ids))/len(cited) if cited else 0.0,
        "relevance_recall": len(relevant & set(retrieved_ids))/len(relevant) if relevant else None,
        "abstention_correct": float("insufficient evidence" in answer.lower()) if example.get("answerable") is False else None,
        "claim_support": None,
    }


def run_rag_ablation(examples, variants, answerer, corpus, seed=2026, usage=None):
    """Retrievers return IDs; the same answerer and final context budget serve every arm."""
    rows = []
    rng = random.Random(seed)
    for index, example in enumerate(examples):
        names = list(variants)
        rng.shuffle(names)
        for name in names:
            started = time.perf_counter()
            usage_start = len(usage) if usage is not None else 0
            error, ids, answer = None, [], ""
            try:
                ids = variants[name](example["question"])
                if not isinstance(ids, list) or any(item not in corpus for item in ids):
                    raise ValueError("Retriever returned unknown chunk IDs")
                answer = answerer(example["question"], ids)
            except Exception as exc:
                error = type(exc).__name__
                ids = [item for item in (ids or []) if item in corpus]
            calls = usage[usage_start:] if usage is not None else []
            rows.append({"example_id": index, "system": name, "question": example["question"],
                         "reference": example["reference"], "answer": answer,
                         "retrieved_chunk_ids": ids, "error": error,
                         "stratum": example.get("stratum", "natural_qa"),
                         "input_tokens": sum(call['input_tokens'] for call in calls),
                         "output_tokens": sum(call['output_tokens'] for call in calls),
                         "generation_calls": len(calls),
                         "latency_seconds": time.perf_counter()-started,
                         **rag_measurements(example, answer, ids, corpus)})
    return rows, summarize_rag_rows(rows, variants, seed)


def summarize_rag_rows(rows, variants, seed=2026):
    summaries = {}
    for name in variants:
        selected = [row for row in rows if row["system"] == name]
        metrics, metric_counts = {}, {}
        for key in ["answer_hit_proxy", "reciprocal_rank_proxy", "token_f1", "binary_answer_correct", "citation_id_validity", "relevance_recall", "abstention_correct"]:
            values = [row[key] for row in selected if row[key] is not None]
            metrics[key] = statistics.fmean(values) if values else None
            metric_counts[key] = len(values)
        times = sorted(row["latency_seconds"] for row in selected)
        summaries[name] = {**metrics, "metric_counts": metric_counts, "examples": len(selected),
                           "generation_calls": sum(row['generation_calls'] for row in selected),
                           "input_tokens": sum(row['input_tokens'] for row in selected),
                           "output_tokens": sum(row['output_tokens'] for row in selected),
                           "errors": sum(row["error"] is not None for row in selected),
                           "latency_p50_seconds": statistics.median(times),
                           "latency_p95_seconds": times[min(len(times)-1, int(.95*len(times)))]}
    baseline = [row["token_f1"] for row in rows if row["system"] == "dense"]
    for name in variants:
        selected = [row["token_f1"] for row in rows if row["system"] == name]
        summaries[name]["paired_f1_vs_dense"] = paired_mean_interval(baseline, selected, seed)
    return summaries


def validate_relevance_labels(labels, corpus):
    if not isinstance(labels, list):
        raise ValueError("Held-out labels must be a list")
    for item in labels:
        if not all(isinstance(item.get(key), str) and item[key].strip()
                   for key in ['question', 'reference', 'label_origin']):
            raise ValueError("Labels require question, reference, and label_origin")
        if type(item.get('answerable')) is not bool:
            raise ValueError("answerable must be an explicit boolean label")
        ids = item.get('relevant_chunk_ids')
        if not isinstance(ids, list) or any(chunk not in corpus for chunk in ids):
            raise ValueError("Unknown or missing relevant_chunk_ids")
        if bool(ids) != item['answerable']:
            raise ValueError("Answerability and relevant evidence labels conflict")
    return [{**item, 'stratum': 'independently_labeled'} for item in labels]


def agent_accounting(result, case):
    planner = result["planner"]
    fallback = bool(planner.get("fallback_used", not planner.get("valid_json", False)))
    routed = result["planned_tool"] == case["tool"]
    return {"fallback_used": fallback, "model_routing_correct": routed and not fallback,
            "model_parse_valid": bool(planner.get("valid_json", False)),
            "planner_attempts": planner.get("attempts", 0),
            "tool_execution_status": result.get("status", "unknown")}


def answer_text(value):
    return str(value[0]) if isinstance(value, (list, tuple)) and value else str(value)

eligible = [
    {"question": str(row["question"]).strip(), "reference": answer_text(row["answer"]).strip()}
    for row in qa_ds if str(row.get("question", "")).strip() and answer_text(row.get("answer", "")).strip()
]
eval_examples = random.Random(SEED).sample(eligible, min(EVAL_SIZE, len(eligible)))
corpus_by_id = {chunk.metadata["chunk_id"]: chunk.page_content for chunk in chunks}
# Optional independently labeled JSON: question, reference, relevant_chunk_ids, answerable.
# Do not label absence merely because a reference substring was not found.
label_path = PROJECT_DIR / "heldout_relevance.json"
if label_path.exists():
    labeled = json.loads(label_path.read_text())
    eval_examples.extend(validate_relevance_labels(labeled, corpus_by_id))

# Authored absence probes are reported separately from public natural QA.
for question in ["What private passphrase did I choose this morning?",
                 "What is the unpublished result of my personal medical test?",
                 "What is the current balance of my private bank account?"]:
    eval_examples.append({'question': question, 'reference': 'Insufficient evidence.',
                          'answerable': False, 'stratum': 'authored_absence_probe'})

corpus_by_id = {chunk.metadata["chunk_id"]: chunk.page_content for chunk in chunks}
position_by_id = {chunk.metadata["chunk_id"]: i for i, chunk in enumerate(chunks)}
from conditional_hyde import retrieve as conditional_retrieve, POLICY as HYDE_POLICY
conditional_hyde_trace = []
def conditional_positions(question):
    return conditional_retrieve(question, dense_rank, sparse_rank, reciprocal_rank_fusion,
        rerank, lambda q: generate_batch([hyde_prompt(q)], max_new_tokens=96)[0],
        CANDIDATE_K, ADVANCED_K, conditional_hyde_trace)

def ids_at(positions):
    return [chunks[i].metadata["chunk_id"] for i in positions]
def hybrid_positions(question):
    return reciprocal_rank_fusion([dense_rank(question, CANDIDATE_K), sparse_rank(question, CANDIDATE_K)], CANDIDATE_K)
def hyde_positions(question):
    hypothetical = generate_batch([hyde_prompt(question)], max_new_tokens=96)[0]
    return advanced_retrieve(question, hypothetical)
variants = {
    "dense": lambda q: ids_at(baseline_retrieve(q)),
    "hybrid": lambda q: ids_at(hybrid_positions(q)[:ADVANCED_K]),
    "hybrid_rerank": lambda q: ids_at(rerank(q, hybrid_positions(q), ADVANCED_K)),
    "hyde_hybrid_rerank": lambda q: ids_at(hyde_positions(q)),
    "conditional_hyde_hybrid_rerank": lambda q: ids_at(conditional_positions(q)),
}
def shared_answerer(question, ids):
    indices = [position_by_id[item] for item in ids]
    return generate_batch([baseline_answer_prompt(question, indices)], max_new_tokens=96)[0]

# Warm each inference path, excluding these examples and timings from the evaluation.
for retrieve in variants.values():
    shared_answerer("What is the purpose of this encyclopedia?", retrieve("What is the purpose of this encyclopedia?"))
conditional_hyde_trace.clear()
records, arm_summaries = run_rag_ablation(eval_examples, variants, shared_answerer, corpus_by_id, SEED, rag_generation_usage)
results = pd.DataFrame(records)
summary = pd.DataFrame(arm_summaries).T
run_summary = {
    "evaluation_version": "upgrade-v2",
    "conditional_hyde_policy": HYDE_POLICY,
    "conditional_hyde_trace": conditional_hyde_trace, "dataset": DATASET_NAME, "dataset_revision": DATASET_REVISION,
    "examples": len(eval_examples), "evaluation_seed": SEED, "generator_model": GENERATOR_MODEL,
    "generator_revision": getattr(generator_model.config, "_commit_hash", None),
    "embedding_fingerprint": embedding_fingerprint,
    "reranker_revision": getattr(rerank_model.config, "_commit_hash", None),
    "answer_prompt_sha256": fingerprint(baseline_answer_prompt("QUESTION", [])),
    "evaluation_sha256": fingerprint(eval_examples),
    "decoding": {"do_sample": False, "max_new_tokens": 96, "input_max_tokens": 3072},
    "context_chunks": ADVANCED_K, "gpu": torch.cuda.get_device_name(0),
    "metrics": arm_summaries,
    "metrics_by_stratum": {name: summarize_rag_rows([row for row in records if row['stratum'] == name], variants, SEED)
                           for name in sorted({row['stratum'] for row in records})},
    "limitations": "Substring hit/MRR and token F1 are proxies. Citation ID validity is not claim support. Human claim-support labels remain pending.",
}
results.to_json(OUT_DIR / "project2_eval_results.json", orient="records", indent=2)
(OUT_DIR / "project2_summary.json").write_text(json.dumps(run_summary, indent=2))
display(summary)
print("Saved versioned ablations:", OUT_DIR)
"""Blind evidence-support review. A valid citation ID is never a support label."""
import hashlib
import json
import random


def evidence_identity(row, corpus):
    evidence = {chunk: corpus[chunk] for chunk in row['retrieved_chunk_ids']}
    payload = [row['question'], row['answer'], evidence]
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()


def grounding_review_pack(rows, corpus, seed=2026):
    shuffled = list(rows)
    random.Random(seed).shuffle(shuffled)
    entries, key = [], {}
    for position, row in enumerate(shuffled):
        review_id = f'review-{position:04d}'
        entries.append({'review_id': review_id, 'identity': evidence_identity(row, corpus),
                        'question': row['question'], 'answer': row['answer'],
                        'retrieved_evidence': {chunk: corpus[chunk] for chunk in row['retrieved_chunk_ids']},
                        'claims': None, 'reviewer': None})
        key[review_id] = {'system': row['system'], 'example_id': row['example_id']}
    return {'rubric_version': 'claim-support-v1', 'instructions': (
        'Split factual assertions into atomic claims. For each claim record text, supported '
        '(boolean), cited_chunk_id, supporting_quote, and rationale. A quote must appear '
        'verbatim in the cited retrieved chunk; judge entailment, not word overlap. '
        'Use claims=[] for a pure abstention and name the reviewer. Leave unreviewed entries null.'),
        'entries': entries}, key


def score_review(review, original):
    if review['identity'] != original['identity']:
        raise ValueError('Stale answer/evidence review')
    claims = review.get('claims')
    if claims is None:
        return {'supported': None, 'claims': None}
    if not review.get('reviewer') or not isinstance(claims, list):
        raise ValueError('Reviewed claims require a reviewer and a list')
    for claim in claims:
        if type(claim.get('supported')) is not bool or not claim.get('text') or not claim.get('rationale'):
            raise ValueError('Each claim requires a decision and reason')
        if claim['supported']:
            quote = claim.get('supporting_quote')
            evidence = original['retrieved_evidence'].get(claim.get('cited_chunk_id'), '')
            if not quote or quote not in evidence:
                raise ValueError('Support must cite an exact quote from retrieved evidence')
    return {'supported': sum(claim['supported'] for claim in claims), 'claims': len(claims)}

review_pack, review_key = grounding_review_pack(records, corpus_by_id, SEED)
(OUT_DIR / 'grounding_review.json').write_text(json.dumps(review_pack, indent=2))
(OUT_DIR / 'grounding_review_key.json').write_text(json.dumps(review_key, indent=2))


## 7. Optional provider benchmark: GPT-4o, LLaMA, and Mistral

This cell reuses the exact advanced retrieved contexts for a fair generator
comparison. It is disabled by default to prevent accidental API charges.
Keys are read from environment variables or hidden `getpass` prompts and are
never written to the notebook. DeepInfra exposes an OpenAI-compatible API.
Set the model IDs to currently available DeepInfra deployments if needed.


In [ ]:
RUN_PROVIDER_BENCHMARKS = False
PROVIDER_EVAL_SIZE = 10

provider_manifest = pd.DataFrame([
    {
        'provider': 'Local',
        'model': GENERATOR_MODEL,
        'status': 'completed in main evaluation',
    },
    {
        'provider': 'OpenAI',
        'model': os.getenv('OPENAI_MODEL', 'gpt-4o'),
        'status': 'set RUN_PROVIDER_BENCHMARKS=True and supply OPENAI_API_KEY',
    },
    {
        'provider': 'DeepInfra',
        'model': os.getenv('DEEPINFRA_LLAMA_MODEL', 'meta-llama/Meta-Llama-3.1-8B-Instruct'),
        'status': 'set RUN_PROVIDER_BENCHMARKS=True and supply DEEPINFRA_API_KEY',
    },
    {
        'provider': 'DeepInfra',
        'model': os.getenv('DEEPINFRA_MISTRAL_MODEL', 'mistralai/Mistral-7B-Instruct-v0.3'),
        'status': 'set RUN_PROVIDER_BENCHMARKS=True and supply DEEPINFRA_API_KEY',
    },
])

if RUN_PROVIDER_BENCHMARKS:
    from getpass import getpass
    from openai import OpenAI

    def obtain_secret(name):
        value = os.getenv(name)
        if not value:
            value = getpass(f'{name} (hidden): ').strip()
        return value

    provider_specs = [
        ('OpenAI', os.getenv('OPENAI_MODEL', 'gpt-4o'), None, obtain_secret('OPENAI_API_KEY')),
        (
            'DeepInfra-LLaMA',
            os.getenv('DEEPINFRA_LLAMA_MODEL', 'meta-llama/Meta-Llama-3.1-8B-Instruct'),
            'https://api.deepinfra.com/v1/openai',
            obtain_secret('DEEPINFRA_API_KEY'),
        ),
        (
            'DeepInfra-Mistral',
            os.getenv('DEEPINFRA_MISTRAL_MODEL', 'mistralai/Mistral-7B-Instruct-v0.3'),
            'https://api.deepinfra.com/v1/openai',
            os.getenv('DEEPINFRA_API_KEY'),
        ),
    ]
    provider_records = []
    benchmark_rows = [row for row in records if row['system'] == 'hybrid_rerank'
                      and row['stratum'] == 'natural_qa'][:PROVIDER_EVAL_SIZE]
    benchmark_prompts = [baseline_answer_prompt(row['question'], [position_by_id[item]
                        for item in row['retrieved_chunk_ids']]) for row in benchmark_rows]
    for provider, model_name, base_url, api_key in provider_specs:
        client = OpenAI(api_key=api_key, base_url=base_url)
        for i, prompt in enumerate(benchmark_prompts):
            started = time.perf_counter()
            response = client.chat.completions.create(
                model=model_name,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=0,
                max_tokens=96,
            )
            answer = response.choices[0].message.content.strip()
            provider_records.append({
                'provider': provider,
                'model': model_name,
                'example_id': i,
                'answer': answer,
                'token_f1': overlap_f1(answer, benchmark_rows[i]['reference']),
                'citation_validity': rag_measurements(benchmark_rows[i], answer, benchmark_rows[i]['retrieved_chunk_ids'], corpus_by_id)['citation_id_validity'],
                'latency_seconds': time.perf_counter() - started,
            })
    provider_results = pd.DataFrame(provider_records)
    provider_results.to_csv(OUT_DIR / 'provider_benchmark.csv', index=False)
    display(provider_results.groupby(['provider', 'model'])[
        ['token_f1', 'citation_validity', 'latency_seconds']
    ].mean().round(4))
else:
    display(provider_manifest)
    print('Provider calls skipped; the local baseline-vs-advanced evaluation is complete.')


In [16]:
PROJECT = 2
BUNDLE_SHA256 = '1f95483ac4e87252273229aa463526dda7fa60d6397e8d11b1f3786eac859387'
BUNDLE = '...'
from datetime import datetime, timezone
import base64
import gzip
import hashlib
import json
import os
from pathlib import Path
import subprocess
import sys

from google.colab import drive

mount = Path("/content/drive")
if not mount.is_mount():
    drive.mount(str(mount), timeout_ms=180000)
if not mount.is_mount():
    raise RuntimeError("Connect this notebook to the mounted MyPro runtime")
gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
if "A100" not in gpu:
    raise RuntimeError(f"Expected the A100, found {gpu}")
if globals().get("inference_worker") is not None and inference_worker.poll() is None:
    raise RuntimeError("This notebook already has an active evaluation worker")
if subprocess.check_output(
    ["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"], text=True).strip():
    raise RuntimeError("Another process is using the GPU")

runtime = Path("/content/portfolio_inference_v2")
bundle_dir = runtime / f"project{PROJECT}" / BUNDLE_SHA256
tools_dir = bundle_dir / "tools"
tools_dir.mkdir(parents=True, exist_ok=True)
bundle = gzip.decompress(base64.b64decode(BUNDLE))
if hashlib.sha256(bundle).hexdigest() != BUNDLE_SHA256:
    raise RuntimeError("Notebook package hash mismatch")
for name, contents in json.loads(bundle).items():
    (tools_dir / name).write_text(contents)

bootstrap = runtime / "bootstrap"
subprocess.run([sys.executable, "-m", "pip", "install", "--target", str(bootstrap),
                "--no-deps", "uv==0.12.17"], check=True, timeout=180)
uv_cache = Path("/content/project1_training_v2/uv-cache")
if not uv_cache.is_dir():
    uv_cache = runtime / "uv-cache"
uv_env = dict(os.environ, PYTHONPATH=str(bootstrap), UV_CACHE_DIR=str(uv_cache),
              UV_HTTP_RETRIES="1")

def uv(*arguments):
    subprocess.run([sys.executable, "-m", "uv", *map(str, arguments)], env=uv_env,
                   check=True, timeout=600)

venv = runtime / "venv"
python = venv / "bin/python"
lock = tools_dir / "requirements-inference-v2.hashes.txt"
fingerprint = hashlib.sha256(lock.read_bytes()).hexdigest()
marker = venv / ".portfolio-lock"
if not python.is_file():
    uv("venv", "--python", "3.11.17", venv)
if not marker.is_file() or marker.read_text() != fingerprint:
    uv("pip", "install", "--python", python, "--require-hashes", "-r", lock)
    uv("pip", "check", "--python", python)
    marker.write_text(fingerprint)

project_root = mount / f"MyDrive/projects/MyProject{PROJECT}"
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_parent = project_root / "outputs/runs/upgrade-v2"
run_parent.mkdir(parents=True, exist_ok=True)
run_dir = run_parent / run_id
inference_log = run_parent / f"{run_id}.log"
inference_env = {
    k: v for k, v in os.environ.items()
    if not any(word in k.upper() for word in ("TOKEN", "SECRET", "PASSWORD", "API_KEY"))
}
inference_env.update(
    HF_HUB_DISABLE_IMPLICIT_TOKEN="1", HF_HUB_DISABLE_XET="1",
    HF_HUB_DISABLE_TELEMETRY="1", PYTHONUNBUFFERED="1", OTEL_SDK_DISABLED="true",
)
with inference_log.open("x") as log:
    inference_worker = subprocess.Popen(
        [str(python), str(tools_dir / "run_inference_cells_v2.py"), "--project",
         str(PROJECT), "--cells", str(tools_dir / "cells.json"), "--output", str(run_dir)],
        env=inference_env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
    )
print({"project": PROJECT, "run_id": run_id, "pid": inference_worker.pid})


{'project': 2, 'run_id': '20261118T092816338337Z', 'pid': 102758}


In [28]:
import json
import subprocess
import time

exit_code = inference_worker.poll()
manifest_path = run_dir / "run_manifest.json"
if manifest_path.is_file():
    print(json.dumps(json.loads(manifest_path.read_text()), indent=2))
print("Worker exit:", exit_code)
if exit_code is None:
    gpu = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader"], text=True)
    elapsed = max(0, time.time() - inference_log.stat().st_ctime)
    print("GPU utilization, allocated memory:", gpu.strip(), "| Elapsed seconds:", round(elapsed))
elif exit_code != 0:
    with inference_log.open("rb") as log:
        log.seek(max(0, inference_log.stat().st_size - 4096))
        print(log.read().decode(errors="replace"))
elif PROJECT == 2:
    summary = json.loads((run_dir / "project2_summary.json").read_text())
    keys = ("token_f1", "generation_calls", "errors", "latency_p50_seconds", "latency_p95_seconds")
    metrics = {name: {key: row[key] for key in keys} for name, row in summary["metrics"].items()}
    print("RAG metrics:", json.dumps({"examples": summary["examples"], "metrics": metrics}))
elif PROJECT == 3:
    summary = json.loads((run_dir / "project3_summary.json").read_text())
    print("Agent evaluation:", json.dumps(summary))


{
  "project": 2,
  "run_id": "20261118T092816338337Z",
  "status": "complete",
  "gpu": "NVIDIA A100-SXM4-40GB",
  "python": "3.11.17",
  "lock_sha256": "afc112d1c55172cee298cc26e4e3119e2e9d8506dda21494050212c69ed3c906",
  "cell_bundle_sha256": "cf5d58d6bdfc84031cfd186556d6e001f41431fa32be471513356365d5b66ee0",
  "completed_cells": [
    3,
    4,
    6,
    7,
    9,
    11,
    13,
    15,
    17
  ],
  "active_cell": 17,
  "seconds": 916.2690874319997,
  "peak_gpu_allocated_bytes": 6289034240
}
Worker exit: 0
RAG metrics: {"examples": 63, "metrics": {"dense": {"token_f1": 0.17775562491726984, "generation_calls": 63, "errors": 0, "latency_p50_seconds": 1.1068132009968394, "latency_p95_seconds": 2.7248758130008355}, "hybrid": {"token_f1": 0.19068446782870782, "generation_calls": 63, "errors": 0, "latency_p50_seconds": 0.9909555810008897, "latency_p95_seconds": 2.799193729999388}, "hybrid_rerank": {"token_f1": 0.1782409142633708, "generation_calls": 63, "errors": 0, "latency_p50_secon

In [47]:
PROJECT = 2
BUNDLE_SHA256 = 'd4eedadcb075516f8ea7e74e90c8962b70ad8aceec3052b4dad4e2b3057a076a'
BUNDLE = '...'
BENCHMARK = 'retriever-refresh'
from datetime import datetime, timezone
import base64
import gzip
import hashlib
import json
import os
from pathlib import Path
import subprocess
import sys

from google.colab import drive

mount = Path("/content/drive")
if not mount.is_mount():
    drive.mount(str(mount), timeout_ms=180000)
if not mount.is_mount():
    raise RuntimeError("Connect this notebook to the mounted MyPro runtime")
gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
if "A100" not in gpu:
    raise RuntimeError(f"Expected the A100, found {gpu}")
if globals().get("inference_worker") is not None and inference_worker.poll() is None:
    raise RuntimeError("This notebook already has an active evaluation worker")
if subprocess.check_output(
    ["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"], text=True).strip():
    raise RuntimeError("Another process is using the GPU")

runtime = Path("/content/portfolio_inference_v2")
bundle_dir = runtime / f"project{PROJECT}" / BUNDLE_SHA256
tools_dir = bundle_dir / "tools"
tools_dir.mkdir(parents=True, exist_ok=True)
bundle = gzip.decompress(base64.b64decode(BUNDLE))
if hashlib.sha256(bundle).hexdigest() != BUNDLE_SHA256:
    raise RuntimeError("Notebook package hash mismatch")
for name, contents in json.loads(bundle).items():
    destination = (tools_dir / name).resolve()
    if not destination.is_relative_to(tools_dir):
        raise ValueError("Unexpected path in notebook package")
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(contents)

bootstrap = runtime / "bootstrap"
subprocess.run([sys.executable, "-m", "pip", "install", "--target", str(bootstrap),
                "--no-deps", "uv==0.12.17"], check=True, timeout=180)
uv_cache = Path("/content/project1_training_v2/uv-cache")
if not uv_cache.is_dir():
    uv_cache = runtime / "uv-cache"
uv_env = dict(os.environ, PYTHONPATH=str(bootstrap), UV_CACHE_DIR=str(uv_cache),
              UV_HTTP_RETRIES="1")

def uv(*arguments):
    subprocess.run([sys.executable, "-m", "uv", *map(str, arguments)], env=uv_env,
                   check=True, timeout=600)

venv = runtime / "venv"
python = venv / "bin/python"
lock = tools_dir / "requirements-inference-v2.hashes.txt"
fingerprint = hashlib.sha256(lock.read_bytes()).hexdigest()
marker = venv / ".portfolio-lock"
if not python.is_file():
    uv("venv", "--python", "3.11.17", venv)
if not marker.is_file() or marker.read_text() != fingerprint:
    uv("pip", "install", "--python", python, "--require-hashes", "-r", lock)
    uv("pip", "check", "--python", python)
    marker.write_text(fingerprint)

project_root = mount / f"MyDrive/projects/MyProject{PROJECT}"
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_parent = project_root / "outputs/runs/upgrade-v2"
run_parent.mkdir(parents=True, exist_ok=True)
run_dir = run_parent / run_id
inference_log = run_parent / f"{run_id}.log"
inference_env = {
    k: v for k, v in os.environ.items()
    if not any(word in k.upper() for word in ("TOKEN", "SECRET", "PASSWORD", "API_KEY"))
}
inference_env.update(
    HF_HUB_DISABLE_IMPLICIT_TOKEN="1", HF_HUB_DISABLE_XET="1",
    HF_HUB_DISABLE_TELEMETRY="1", PYTHONUNBUFFERED="1", OTEL_SDK_DISABLED="true",
)
with inference_log.open("x") as log:
    benchmark_args = ["--benchmark", BENCHMARK] if BENCHMARK else []
    inference_worker = subprocess.Popen(
        [str(python), str(tools_dir / "run_inference_cells_v2.py"), "--project",
         str(PROJECT), "--cells", str(tools_dir / "cells.json"), "--output", str(run_dir), *benchmark_args],
        env=inference_env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
    )
print({"project": PROJECT, "run_id": run_id, "pid": inference_worker.pid})


{'project': 2, 'run_id': '20261118T102500977498Z', 'pid': 119840}


In [49]:
from pathlib import Path
import json

manifest_path = run_dir / "run_manifest.json"
if manifest_path.is_file():
    print(json.dumps(json.loads(manifest_path.read_text()), indent=2))
print("Worker exit:", inference_worker.poll())
if inference_worker.poll() not in (None, 0):
    with inference_log.open("rb") as log:
        log.seek(max(0, inference_log.stat().st_size - 4096))
        print(log.read().decode(errors="replace"))
elif inference_worker.poll() == 0:
    summary = json.loads((run_dir / "retriever_refresh_summary.json").read_text())
    print("Retriever refresh:", json.dumps(summary))


{
  "project": 2,
  "run_id": "20261118T102500977498Z",
  "status": "complete",
  "gpu": "NVIDIA A100-SXM4-40GB",
  "python": "3.11.17",
  "lock_sha256": "afc112d1c55172cee298cc26e4e3119e2e9d8506dda21494050212c69ed3c906",
  "cell_bundle_sha256": "9db118d4d185a9b920cac8b00e8e2d97ee3bc3073a1bd0a718dc152a557eb631",
  "completed_cells": [
    3,
    4,
    6,
    7,
    9,
    11,
    15
  ],
  "benchmark": "retriever-refresh",
  "active_cell": 15,
  "seconds": 58.47834340999907,
  "peak_gpu_allocated_bytes": 3917818368
}
Worker exit: 0
Retriever refresh: {"cases": 28, "natural_holdout_cases": 60, "holdout_run": "20261118T092816338337Z", "metrics": {"e5_hybrid": {"answer_hit_at_5_proxy": 0.7142857142857143, "mrr_at_5_proxy": 0.6232142857142857, "paired_hit_change": {"mean_change": 0.0, "ci95": [0.0, 0.0], "pairs": 28, "resamples": 2000}, "p50_seconds": 0.014838978500847588, "p95_seconds": 0.022923963999346597, "errors": 0}, "qwen3_hybrid": {"answer_hit_at_5_proxy": 0.7142857142857143, "mrr